# 02 — Prétraitement GPX V2
## Validation, distance cumulée et rééchantillonnage spatial à 10 m

Ce notebook constitue la brique déterministe commune du pipeline.

```text
GPX réel
   ↓
validation de la trace
   ↓
nettoyage
   ↓
distance cumulée
   ↓
rééchantillonnage spatial exact
   ↓
route_resampled.csv
   ↓
   ├── 03 — Agent IA 1 : choix de fenêtre / analyse géométrique
   └── 03bis — Agent IA 2 : environnement / forêt
```

### Changement important par rapport à la V1

**Il n'existe plus de route synthétique de secours.**

Le notebook exige un vrai fichier `.gpx` dans :

```text
data/input/
```

Il est conçu pour être exécuté simplement **du haut vers le bas**, cellule après cellule.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import gpxpy
import matplotlib.pyplot as plt

## 1. Configuration

`BASE_SPACING_M = 10.0` est la **résolution spatiale de base** de la route.

Elle ne doit pas être confondue avec les fenêtres d'analyse choisies plus tard
par l'Agent IA du notebook 03.

Si plusieurs segments GPX existent, on accepte leur raccordement uniquement si
la distance entre la fin d'un segment et le début du suivant reste raisonnable.
Sinon le notebook s'arrête pour éviter d'inventer une liaison inexistante.

In [ ]:
INPUT_DIR = Path("data/input")
OUTPUT_DIR = Path("data/output")

BASE_SPACING_M = 10.0

# Au-delà de cette distance entre deux segments GPX successifs,
# on considère que la trace est potentiellement discontinue.
MAX_SEGMENT_GAP_M = 50.0

# False = comportement sûr : arrêt si un grand trou est détecté.
# True = autorise malgré tout la connexion des segments.
ALLOW_DISCONNECTED_SEGMENTS = False

# Le pipeline pente du notebook 03 nécessite une altitude exploitable.
REQUIRE_ELEVATION = True

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier d'entrée :", INPUT_DIR.resolve())
print("Dossier de sortie :", OUTPUT_DIR.resolve())
print("Pas spatial de base :", BASE_SPACING_M, "m")

## 2. Trouver automatiquement le GPX

Pour permettre une exécution directe du haut vers le bas :

- s'il existe **un seul** fichier `.gpx` dans `data/input/`, il est utilisé ;
- s'il n'y en a aucun, le notebook s'arrête ;
- s'il y en a plusieurs, le notebook s'arrête pour éviter de choisir arbitrairement.

Il suffit donc, pour un test simple, de déposer **un seul GPX** dans `data/input/`.

In [ ]:
gpx_files = sorted(INPUT_DIR.glob("*.gpx"))

if len(gpx_files) == 0:
    raise FileNotFoundError(
        "Aucun fichier .gpx trouvé dans data/input/. "
        "Ajoute un vrai GPX puis relance le notebook depuis le début."
    )

if len(gpx_files) > 1:
    names = [p.name for p in gpx_files]
    raise RuntimeError(
        "Plusieurs fichiers GPX sont présents dans data/input/. "
        "Garde un seul fichier pour une exécution automatique. "
        f"Fichiers trouvés : {names}"
    )

gpx_path = gpx_files[0]

print("GPX sélectionné :", gpx_path)

## 3. Distance entre deux coordonnées

On conserve la formule de Haversine pour calculer les petites distances successives
le long de l'itinéraire.

Pour des itinéraires de quelques kilomètres, cette approximation est adaptée à
notre besoin de distance curviligne de travail.

In [ ]:
EARTH_RADIUS_M = 6_371_000.0


def haversine_m(lat1, lon1, lat2, lon2):
    phi1 = math.radians(float(lat1))
    phi2 = math.radians(float(lat2))

    dphi = math.radians(float(lat2) - float(lat1))
    dlambda = math.radians(float(lon2) - float(lon1))

    a = (
        math.sin(dphi / 2.0) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2.0) ** 2
    )

    # Protection contre les petites erreurs numériques.
    a = min(1.0, max(0.0, a))

    c = 2.0 * math.atan2(
        math.sqrt(a),
        math.sqrt(1.0 - a),
    )

    return EARTH_RADIUS_M * c

## 4. Lire la structure du GPX

On conserve les informations de provenance de chaque point :

```text
track_id
segment_id
point_id
```

Cela permet de détecter les ruptures entre segments au lieu de les concaténer
silencieusement.

In [ ]:
with gpx_path.open("r", encoding="utf-8") as f:
    gpx = gpxpy.parse(f)

rows = []
global_segment_id = 0

for track_id, track in enumerate(gpx.tracks):
    for segment_id_in_track, segment in enumerate(track.segments):

        for point_id, point in enumerate(segment.points):
            rows.append({
                "track_id": int(track_id),
                "segment_id": int(global_segment_id),
                "segment_id_in_track": int(segment_id_in_track),
                "point_id": int(point_id),
                "lat": float(point.latitude),
                "lon": float(point.longitude),
                "elev_m": (
                    float(point.elevation)
                    if point.elevation is not None
                    else np.nan
                ),
            })

        global_segment_id += 1

if len(rows) < 2:
    raise ValueError(
        "Le GPX ne contient pas assez de points de trace exploitables."
    )

route_raw = pd.DataFrame(rows)

n_tracks = len(gpx.tracks)
n_segments = route_raw["segment_id"].nunique()

print("Tracks :", n_tracks)
print("Segments :", n_segments)
print("Points bruts :", len(route_raw))

display(route_raw.head())

## 5. Contrôles qualité de base

On vérifie notamment :

- latitude dans `[-90, 90]` ;
- longitude dans `[-180, 180]` ;
- nombre d'altitudes manquantes ;
- points consécutifs identiques ou quasiment identiques.

In [ ]:
if not route_raw["lat"].between(-90.0, 90.0).all():
    raise ValueError("Le GPX contient au moins une latitude invalide.")

if not route_raw["lon"].between(-180.0, 180.0).all():
    raise ValueError("Le GPX contient au moins une longitude invalide.")

missing_elevation_raw = int(route_raw["elev_m"].isna().sum())

print(
    "Altitudes manquantes :",
    missing_elevation_raw,
    "/",
    len(route_raw),
)

# Suppression des doublons consécutifs quasi identiques.
keep_mask = np.ones(len(route_raw), dtype=bool)

duplicates_removed = 0

for i in range(1, len(route_raw)):
    # On ne compare des doublons que dans le même segment.
    if (
        route_raw.loc[i, "segment_id"]
        == route_raw.loc[i - 1, "segment_id"]
    ):
        d = haversine_m(
            route_raw.loc[i - 1, "lat"],
            route_raw.loc[i - 1, "lon"],
            route_raw.loc[i, "lat"],
            route_raw.loc[i, "lon"],
        )

        if d < 0.01:
            keep_mask[i] = False
            duplicates_removed += 1

route_clean = (
    route_raw.loc[keep_mask]
    .reset_index(drop=True)
    .copy()
)

if len(route_clean) < 2:
    raise ValueError(
        "Après nettoyage, la trace contient moins de deux points."
    )

print("Doublons consécutifs supprimés :", duplicates_removed)
print("Points après nettoyage :", len(route_clean))

## 6. Vérifier les raccordements entre segments

Un GPX peut être découpé en plusieurs segments à cause d'une pause d'enregistrement.

Un petit découpage ne pose pas de problème.

En revanche, si :

```text
fin segment A ----------------------- début segment B
                     grand trou
```

il ne faut pas créer artificiellement une route continue.

Le notebook calcule donc chaque distance de raccordement.

In [ ]:
segment_ids = route_clean["segment_id"].drop_duplicates().tolist()

segment_gap_rows = []

for previous_id, next_id in zip(
    segment_ids[:-1],
    segment_ids[1:],
):
    previous_segment = route_clean[
        route_clean["segment_id"] == previous_id
    ]

    next_segment = route_clean[
        route_clean["segment_id"] == next_id
    ]

    if previous_segment.empty or next_segment.empty:
        continue

    p1 = previous_segment.iloc[-1]
    p2 = next_segment.iloc[0]

    gap_m = haversine_m(
        p1["lat"],
        p1["lon"],
        p2["lat"],
        p2["lon"],
    )

    segment_gap_rows.append({
        "previous_segment_id": int(previous_id),
        "next_segment_id": int(next_id),
        "gap_m": float(gap_m),
    })

segment_gaps = pd.DataFrame(segment_gap_rows)

if not segment_gaps.empty:
    display(segment_gaps)

    max_gap_observed_m = float(
        segment_gaps["gap_m"].max()
    )

    problematic = segment_gaps[
        segment_gaps["gap_m"] > MAX_SEGMENT_GAP_M
    ]

    if not problematic.empty:
        message = (
            "Le GPX contient au moins une rupture importante entre "
            "deux segments. Le notebook refuse de fabriquer une liaison "
            f"supérieure à {MAX_SEGMENT_GAP_M:.1f} m. "
            f"Ruptures détectées : "
            f"{problematic.to_dict(orient='records')}"
        )

        if ALLOW_DISCONNECTED_SEGMENTS:
            print("ATTENTION :", message)
            print(
                "ALLOW_DISCONNECTED_SEGMENTS=True : "
                "la connexion est néanmoins autorisée."
            )
        else:
            raise RuntimeError(
                message
                + " Si cette connexion est réellement souhaitée, "
                "passe ALLOW_DISCONNECTED_SEGMENTS=True dans la configuration."
            )
else:
    max_gap_observed_m = 0.0
    print("Un seul segment exploitable : aucun raccordement à vérifier.")

print(
    "Plus grand raccordement observé :",
    round(max_gap_observed_m, 2),
    "m",
)

## 7. Calculer la distance cumulée

La distance est calculée sur la trace nettoyée.

Si plusieurs petits segments sont présents et validés, leur raccordement spatial est
inclus dans la distance cumulée.

In [ ]:
cumulative_distance = [0.0]

for i in range(1, len(route_clean)):
    step_m = haversine_m(
        route_clean.loc[i - 1, "lat"],
        route_clean.loc[i - 1, "lon"],
        route_clean.loc[i, "lat"],
        route_clean.loc[i, "lon"],
    )

    cumulative_distance.append(
        cumulative_distance[-1] + step_m
    )

route_clean["distance_m"] = cumulative_distance

route_length_m = float(
    route_clean["distance_m"].iloc[-1]
)

if route_length_m <= 0:
    raise ValueError(
        "La longueur calculée de la trace est nulle."
    )

print("Longueur de la trace :", round(route_length_m, 1), "m")

## 8. Vérifier et compléter l'altitude

Le notebook 03 analyse notamment les pentes.

Si quelques altitudes sont manquantes, elles sont interpolées à partir de la distance
cumulée.

Si pratiquement aucune altitude n'est disponible et que
`REQUIRE_ELEVATION=True`, le notebook s'arrête.

In [ ]:
valid_elevation = route_clean["elev_m"].notna()

n_valid_elevation = int(valid_elevation.sum())

if n_valid_elevation < 2:
    if REQUIRE_ELEVATION:
        raise ValueError(
            "Le GPX ne contient pas assez d'altitudes pour le pipeline "
            "de détection de pente du notebook 03."
        )
    else:
        print(
            "ATTENTION : altitude insuffisante. "
            "Les analyses de pente ne seront pas exploitables."
        )

elevations_interpolated = 0

if 2 <= n_valid_elevation < len(route_clean):
    distances = route_clean["distance_m"].to_numpy(
        dtype=float
    )

    elevations = route_clean["elev_m"].to_numpy(
        dtype=float
    )

    valid = ~np.isnan(elevations)

    interpolated = np.interp(
        distances,
        distances[valid],
        elevations[valid],
    )

    elevations_interpolated = int(
        np.isnan(elevations).sum()
    )

    route_clean["elev_m"] = interpolated

    print(
        "Altitudes interpolées :",
        elevations_interpolated,
    )
else:
    print("Interpolation d'altitude non nécessaire.")

## 9. Rééchantillonnage spatial exact

La route est interpolée tous les :

```text
10 m
```

mais le **dernier point est forcé à être exactement la fin réelle du GPX**.

Exemple pour une route de 1007,3 m :

```text
0
10
20
...
1000
1007,3
```

On n'ajoute jamais artificiellement un point à 1010 m.

In [ ]:
def build_resampling_distances(
    total_length_m,
    spacing_m,
):
    if spacing_m <= 0:
        raise ValueError(
            "spacing_m doit être strictement positif."
        )

    total_length_m = float(total_length_m)

    regular = np.arange(
        0.0,
        total_length_m,
        float(spacing_m),
        dtype=float,
    )

    if len(regular) == 0:
        regular = np.array([0.0])

    if not np.isclose(
        regular[-1],
        total_length_m,
        atol=1e-9,
    ):
        regular = np.append(
            regular,
            total_length_m,
        )
    else:
        regular[-1] = total_length_m

    return regular


new_distance = build_resampling_distances(
    route_length_m,
    BASE_SPACING_M,
)

raw_distance = route_clean[
    "distance_m"
].to_numpy(dtype=float)

route = pd.DataFrame({
    "distance_m": new_distance,
    "lat": np.interp(
        new_distance,
        raw_distance,
        route_clean["lat"].to_numpy(
            dtype=float
        ),
    ),
    "lon": np.interp(
        new_distance,
        raw_distance,
        route_clean["lon"].to_numpy(
            dtype=float
        ),
    ),
    "elev_m": np.interp(
        new_distance,
        raw_distance,
        route_clean["elev_m"].to_numpy(
            dtype=float
        ),
    ),
})

print("Points rééchantillonnés :", len(route))
print(
    "Dernière distance :",
    route["distance_m"].iloc[-1],
    "m",
)

print(
    "Longueur GPX nettoyé :",
    route_length_m,
    "m",
)

assert np.isclose(
    route["distance_m"].iloc[-1],
    route_length_m,
)

display(route.head())
display(route.tail())

## 10. Contrôler le pas spatial obtenu

In [ ]:
resampled_steps = np.diff(
    route["distance_m"].to_numpy(
        dtype=float
    )
)

if len(resampled_steps) > 0:
    regular_steps = resampled_steps[:-1]

    if len(regular_steps) > 0:
        print(
            "Pas régulier moyen :",
            round(float(regular_steps.mean()), 6),
            "m",
        )

        assert np.allclose(
            regular_steps,
            BASE_SPACING_M,
            atol=1e-6,
        )

    print(
        "Dernier intervalle :",
        round(float(resampled_steps[-1]), 6),
        "m",
    )

    if resampled_steps[-1] > BASE_SPACING_M + 1e-6:
        raise AssertionError(
            "Le dernier intervalle est supérieur au pas de base."
        )

print("Validation du rééchantillonnage : OK")

## 11. Profil altimétrique

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    route["distance_m"],
    route["elev_m"],
)

ax.set_xlabel("Distance le long de l'itinéraire (m)")
ax.set_ylabel("Altitude (m)")
ax.set_title("Profil altimétrique de la route normalisée")
ax.grid(True, alpha=0.25)

plt.show()

## 12. Rapport qualité

Le rapport permet de tracer exactement ce qui a été fait avant les expériences
des notebooks suivants.

In [ ]:
final_last_step_m = (
    float(resampled_steps[-1])
    if len(resampled_steps) > 0
    else 0.0
)

quality_report = {
    "input_file": str(gpx_path),
    "n_tracks": int(n_tracks),
    "n_segments": int(n_segments),
    "n_raw_points": int(len(route_raw)),
    "n_clean_points": int(len(route_clean)),
    "duplicates_removed": int(duplicates_removed),
    "missing_elevation_raw": int(
        missing_elevation_raw
    ),
    "elevations_interpolated": int(
        elevations_interpolated
    ),
    "max_segment_gap_m": float(
        max_gap_observed_m
    ),
    "allowed_segment_gap_m": float(
        MAX_SEGMENT_GAP_M
    ),
    "route_length_m": float(
        route_length_m
    ),
    "base_spacing_m": float(
        BASE_SPACING_M
    ),
    "n_resampled_points": int(
        len(route)
    ),
    "final_interval_m": float(
        final_last_step_m
    ),
}

print(
    json.dumps(
        quality_report,
        ensure_ascii=False,
        indent=2,
    )
)

## 13. Export

Trois fichiers sont produits :

```text
data/output/route_resampled.csv
data/output/route_raw_clean.csv
data/output/route_quality.json
```

### Fichier principal

`route_resampled.csv` est l'entrée commune des notebooks :

```text
03
03bis
```

Format :

```text
distance_m | lat | lon | elev_m
```

In [ ]:
route_path = (
    OUTPUT_DIR / "route_resampled.csv"
)

raw_clean_path = (
    OUTPUT_DIR / "route_raw_clean.csv"
)

quality_path = (
    OUTPUT_DIR / "route_quality.json"
)

route.to_csv(
    route_path,
    index=False,
)

route_clean.to_csv(
    raw_clean_path,
    index=False,
)

with quality_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        quality_report,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Exports terminés :")
print("-", route_path)
print("-", raw_clean_path)
print("-", quality_path)

# Sortie du notebook 02 V2

À ce stade :

```text
GPX réel
    ↓
contrôlé et nettoyé
    ↓
route normalisée à 10 m
    ↓
data/output/route_resampled.csv
    ↓
    ├── 03 — Agent IA 1
    └── 03bis — Agent IA 2
```

Il n'est pas nécessaire de conserver ce kernel ouvert pour exécuter les notebooks
suivants : toutes les données nécessaires sont persistées sur disque.